# Random forests

A single [decision tree](../04-trees/decision-trees.ipynb) is **high variance**:
small changes in the training data can change it a lot, so it tends to overfit.
A **random forest** fixes this by training many trees on different bootstrap
samples and averaging their votes — the errors cancel out.

We use `smartcore`'s `RandomForestClassifier` and `RandomForestRegressor`.

In [ ]:
:dep smartcore = { version = "0.3" }
use smartcore::linalg::basic::matrix::DenseMatrix;

// HARD 2-class data: overlapping classes + ~20% label noise, plus a pure-noise
// second feature. This is where a single (deep) tree overfits and a forest helps.
let rows: Vec<Vec<f64>> = (0..300).map(|i| {
    let base = (i % 2) as f64 + 2.0;                     // centres 2 and 3 (overlap)
    let jitter = (((i * 37) % 20) as f64 - 10.0) * 0.12; // +/- ~1.2
    vec![base + jitter, ((i * 53 % 20) as f64 - 10.0) * 0.2]  // 2nd feature = noise
}).collect();
let y: Vec<i64> = (0..300).map(|i| {
    let c = (i % 2) as i64;
    if (i * 41) % 100 < 20 { 1 - c } else { c }          // flip ~20% of labels
}).collect();

fn matrix(rows: &[Vec<f64>]) -> DenseMatrix<f64> {
    DenseMatrix::new(rows.len(), rows[0].len(), rows.iter().flatten().cloned().collect(), false)
}
println!("{} samples, {} features (overlapping classes, ~20% label noise)", rows.len(), rows[0].len());

## A single tree is unstable

Train five decision trees on five different bootstrap resamples and see how often
they *disagree* on the same points — that disagreement is the variance a forest
smooths away:

In [ ]:
{
    use smartcore::api::SupervisedEstimator;
    use smartcore::tree::decision_tree_classifier::DecisionTreeClassifier;

    let n = rows.len();
    // Train 5 trees on 5 reproducible bootstrap resamples; collect their predictions.
    let mut all_preds: Vec<Vec<i64>> = Vec::new();
    for seed in 0..5u64 {
        let mut s = seed.wrapping_mul(6364136223846793005).wrapping_add(1);
        let idx: Vec<usize> = (0..n).map(|_| { s = s.wrapping_mul(6364136223846793005).wrapping_add(1); (s >> 33) as usize % n }).collect();
        let br: Vec<Vec<f64>> = idx.iter().map(|&i| rows[i].clone()).collect();
        let by: Vec<i64> = idx.iter().map(|&i| y[i]).collect();
        let tree = DecisionTreeClassifier::fit(&matrix(&br), &by, Default::default()).unwrap();
        all_preds.push(tree.predict(&matrix(&rows)).unwrap());
    }
    // Fraction of points where the 5 trees don't all agree.
    let disagree = (0..n).filter(|&i| { let v = all_preds[0][i]; !all_preds.iter().all(|p| p[i] == v) }).count();
    println!("points where the 5 bootstrap trees disagree: {} / {} ({:.0}%)", disagree, n, disagree as f64 / n as f64 * 100.0);
}

## The forest is more stable — and usually more accurate

Split the data, then compare a single tree against a random forest on the *same*
held-out test set. The forest's real guarantee is **lower variance** (it averages
away the disagreement we just measured); on noisy data that also tends to show up
as **equal-or-better accuracy**, though the exact gap depends on the dataset:

In [ ]:
{
    use smartcore::api::SupervisedEstimator;
    use smartcore::model_selection::train_test_split;
    use smartcore::metrics::accuracy;
    use smartcore::tree::decision_tree_classifier::DecisionTreeClassifier;
    use smartcore::ensemble::random_forest_classifier::RandomForestClassifier;

    let (xtr, xte, ytr, yte) = train_test_split(&matrix(&rows), &y, 0.3, true, Some(42));
    let tree = DecisionTreeClassifier::fit(&xtr, &ytr, Default::default()).unwrap();
    let forest = RandomForestClassifier::fit(&xtr, &ytr, Default::default()).unwrap();
    println!("single tree    test accuracy = {:.3}", accuracy(&yte, &tree.predict(&xte).unwrap()));
    println!("random forest  test accuracy = {:.3}", accuracy(&yte, &forest.predict(&xte).unwrap()));
}

## Forests for regression too

The existing [Trees chapter](../04-trees/decision-trees.ipynb) did classification;
forests work just as well for regression (averaging numeric predictions):

In [ ]:
{
    use smartcore::api::SupervisedEstimator;
    use smartcore::ensemble::random_forest_regressor::RandomForestRegressor;

    // y = 2*x0 + x1 + noise
    let rx: Vec<Vec<f64>> = (0..120).map(|i| vec![(i % 12) as f64, (i % 7) as f64]).collect();
    let ry: Vec<f64> = rx.iter().enumerate().map(|(i, r)| 2.0 * r[0] + r[1] + ((i % 5) as f64 - 2.0) * 0.3).collect();
    let x = DenseMatrix::new(rx.len(), rx[0].len(), rx.iter().flatten().cloned().collect(), false);
    let forest = RandomForestRegressor::fit(&x, &ry, Default::default()).unwrap();
    let pred = forest.predict(&x).unwrap();
    let rmse = (ry.iter().zip(&pred).map(|(a, p)| (a - p).powi(2)).sum::<f64>() / ry.len() as f64).sqrt();
    println!("random forest regressor RMSE = {:.3}", rmse);
}

## Feature importance

Which features drive the forest? We reuse the model-agnostic **permutation
importance** from the [Explainability chapter](../07-explainability/model-interpretability.ipynb)
(scramble a feature, measure the accuracy drop):

In [ ]:
{
    use smartcore::api::SupervisedEstimator;
    use smartcore::metrics::accuracy;
    use smartcore::ensemble::random_forest_classifier::RandomForestClassifier;

    let forest = RandomForestClassifier::fit(&matrix(&rows), &y, Default::default()).unwrap();
    let base = accuracy(&y, &forest.predict(&matrix(&rows)).unwrap());
    println!("baseline accuracy = {:.3}", base);
    for j in 0..rows[0].len() {
        let mut permuted = rows.clone();
        let col: Vec<f64> = rows.iter().rev().map(|r| r[j]).collect();
        for (i, r) in permuted.iter_mut().enumerate() { r[j] = col[i]; }
        let acc = accuracy(&y, &forest.predict(&matrix(&permuted)).unwrap());
        println!("  feature {} importance (accuracy drop) = {:.3}", j, base - acc);
    }
}

```{note}
**Free parallelism & an ecosystem gap.** `smartcore` builds a forest's trees in
parallel with `rayon` internally — the concrete payoff of the
[Multithreading chapter](../04b-multithreading/parallel-ml.ipynb), with no
`par_iter` on your part. Note also that `smartcore` 0.3 does **not** ship an
Extremely-Randomized-Trees ("Extra Trees") estimator, unlike scikit-learn — a
real gap; if you need it you'd hand-roll the extra randomization.
```

Next: [gradient boosting](gradient-boosting.ipynb) — a *different* way to combine
trees, building them sequentially to correct each other's errors.